In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CONTROL_TABLE = (
    "target_catalog.control.ingestion_control"
)

# Read all active table configurations
# control_df = (
#     spark.table(CONTROL_TABLE)
#     .filter(F.col("active") == True)
# )
control_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("active") == True)
        & (F.upper(F.col("load_type")) == "INCREMENTAL")
    )
)

control_rows = control_df.collect()

control_delta = DeltaTable.forName(
    spark, CONTROL_TABLE
)

print(f"Tables to process: {len(control_rows)}")


for config in control_rows:

    table_id = config["table_id"]
    source_table = config["source_table"]
    target_table = config["target_table"]
    primary_key = config["primary_key"]
    watermark_column = config["watermark_column"]
    last_processed = config["last_processed_value"]

    print(f"\nProcessing: {table_id}")

    try:

        # 1. Read source table
        source_df = spark.table(source_table)

        # 2. Find records newer than the saved watermark
        incremental_df = source_df.filter(
            F.col(watermark_column).isNotNull()
            & (
                F.col(watermark_column)
                > F.lit(last_processed)
            )
        )

        # 3. Keep the latest version of each primary key
        window_spec = (
            Window
            .partitionBy(primary_key)
            .orderBy(
                F.col(watermark_column).desc()
            )
        )

        incremental_df = (
            incremental_df
            .withColumn(
                "_rn",
                F.row_number().over(window_spec)
            )
            .filter(F.col("_rn") == 1)
            .drop("_rn")
        )

        # Keep rescued data in the raw layer;
        # do not carry this operational column into curated data.
        if "_rescued_data" in incremental_df.columns:
            incremental_df = incremental_df.drop(
                "_rescued_data"
            )

        # 4. Count the records going to MERGE
        row_count = incremental_df.count()

        print(
            f"{table_id}: {row_count} incremental rows"
        )

        # 5. No new records
        if row_count == 0:

            control_delta.update(
                condition=(
                    F.col("table_id") == table_id
                ),
                set={
                    "last_run_at": F.current_timestamp(),
                    "rows_processed": F.lit(0).cast("bigint"),
                    "status": F.lit("NO_NEW_DATA"),
                    "error_message": F.lit(None).cast("string")
                }
            )

            print(f"{table_id}: NO_NEW_DATA")
            continue

        # 6. Find the latest processed source timestamp
        max_watermark = (
            incremental_df
            .agg(
                F.max(watermark_column).alias("max_value")
            )
            .first()["max_value"]
        )

        # 7. Create target on the initial load.
        #    Otherwise, update/insert using Delta MERGE.
        target_exists = spark.catalog.tableExists(
            target_table
        )

        if not target_exists:

            (
                incremental_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
            )

            print(
                f"{target_table}: target created"
            )

        else:

            target_delta = DeltaTable.forName(
                spark, target_table
            )

            (
                target_delta.alias("t")
                .merge(
                    incremental_df.alias("s"),
                    f"t.`{primary_key}` = s.`{primary_key}`"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

            print(
                f"{target_table}: MERGE completed"
            )

        # 8. Save the new checkpoint and run status
        control_delta.update(
            condition=(
                F.col("table_id") == table_id
            ),
            set={
                "last_processed_value":
                    F.lit(max_watermark).cast("timestamp"),
                "last_run_at": F.current_timestamp(),
                "rows_processed":
                    F.lit(row_count).cast("bigint"),
                "status": F.lit("SUCCESS"),
                "error_message": F.lit(None).cast("string")
            }
        )

        print(
            f"{table_id}: SUCCESS; "
            f"{row_count} rows processed"
        )

    except Exception as e:

        error_text = str(e)[:4000]

        print(
            f"{table_id}: FAILED - {error_text}"
        )

        control_delta.update(
            condition=(
                F.col("table_id") == table_id
            ),
            set={
                "last_run_at": F.current_timestamp(),
                "rows_processed": F.lit(0).cast("bigint"),
                "status": F.lit("FAILED"),
                "error_message": F.lit(error_text)
            }
        )

print("\nIncremental processing completed.")